# Motion, Reward, and Circuits — Kinematics, RL, and Control
Rigid body motion → Markov decision processes → optimal control → circuit implementation.

In [ ]:
from sympy import *
from IPython.display import display, Markdown
import numpy as np

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Rigid Body Kinematics: Position, Velocity, Acceleration

In [ ]:
section('Kinematics')
t = symbols('t', real=True)

# Position vector
x = Function('x')(t)
y = Function('y')(t)
z_pos = Function('z')(t)
r = Matrix([x, y, z_pos])
step('Position r(t) =', r)

v = r.diff(t)
step('Velocity v = dr/dt =', v)

a = v.diff(t)
step('Acceleration a = dv/dt =', a)

# Projectile — dog jumping
g, v0, theta = symbols('g v_0 theta', positive=True)
x_proj = v0*cos(theta)*t
y_proj = v0*sin(theta)*t - Rational(1,2)*g*t**2
step('Projectile x(t) =', x_proj)
step('Projectile y(t) =', y_proj)

# Range
t_land = solve(y_proj, t)
t_flight = [s for s in t_land if s != 0][0]
R = x_proj.subs(t, t_flight)
step('Time of flight t_f =', t_flight)
step('Range R =', simplify(R))
step('Optimal angle for max range:', solve(diff(R, theta), theta))

## §2 — Rotation: SO(3), Euler Angles, Angular Velocity

In [ ]:
section('Rotation Groups')

phi, psi, theta_e = symbols('phi psi theta', real=True)

# Rotation matrices
Rx = Matrix([
    [1,        0,         0],
    [0,  cos(phi), -sin(phi)],
    [0,  sin(phi),  cos(phi)]
])
Ry = Matrix([
    [ cos(theta_e), 0, sin(theta_e)],
    [0,             1, 0           ],
    [-sin(theta_e), 0, cos(theta_e)]
])
Rz = Matrix([
    [cos(psi), -sin(psi), 0],
    [sin(psi),  cos(psi), 0],
    [0,         0,        1]
])
step('R_x(φ) =', Rx)
step('R_z(ψ) =', Rz)

# Orthogonality
step('R_x · R_xᵀ = I (verify):', simplify(Rx * Rx.T))
step('det(R_x) =', det(Rx).simplify())

# ZYX Euler: body-frame rotation
R_euler = Rz * Ry * Rx
step('ZYX Euler R = R_z R_y R_x: (symbolic, shown for θ=φ=ψ=0)')
step('R|₀ =', R_euler.subs([(phi,0),(theta_e,0),(psi,0)]))

display(Markdown(r'''
**Angular velocity** $\boldsymbol{\omega}$ relates to Euler angle rates:
$$\boldsymbol{\omega} = \dot{\phi}\hat{x} + \dot{\theta}\hat{y} + \dot{\psi}\hat{z}$$
(in body frame, for ZYX convention — used in quadruped motion capture)
'''))

## §3 — Markov Decision Process: Long-Term Reward

In [ ]:
section('Markov Decision Process (MDP)')

display(Markdown(r'''
An MDP is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

- $\mathcal{S}$ — state space (joint angles, velocities, contact forces)
- $\mathcal{A}$ — action space (motor torques)
- $P(s'|s,a)$ — transition probability
- $R(s,a)$ — immediate reward
- $\gamma \in [0,1)$ — discount factor (how much future reward matters)
'''))

gamma, r_t = symbols('gamma r', real=True)
t_sym = symbols('t', integer=True, nonnegative=True)

# Discounted return
G = Sum(gamma**t_sym * r_t, (t_sym, 0, oo))
step('Discounted return G =', G)

# Geometric series bound
R_max = symbols('R_max', positive=True)
G_bound = R_max / (1 - gamma)
step('Upper bound |G| ≤ R_max/(1−γ) =', G_bound)
step('At γ=0.99, R_max=1: G ≤', G_bound.subs([(R_max,1),(gamma,Rational(99,100))]))

# Bellman equation
display(Markdown(r'''
**Bellman optimality equation** — the key recursion:
$$V^*(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s') \right]$$

**Q-function** (action-value):
$$Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^*(s',a')$$

Optimal policy: $\pi^*(s) = \arg\max_a Q^*(s,a)$
'''))

## §4 — Q-Learning and TD Updates

In [ ]:
section('Temporal Difference Learning')

alpha, Q_sa, R_sym, Q_next = symbols('alpha Q R Q_next', real=True)

# TD error
delta = R_sym + gamma*Q_next - Q_sa
step('TD error δ = R + γ·Q(s\',a*) − Q(s,a) =', delta)

# Q update
Q_new = Q_sa + alpha*delta
step('Q update: Q(s,a) ← Q(s,a) + α·δ =', Q_new)

display(Markdown(r'''
**Convergence**: Q-learning converges to $Q^*$ if:
1. Every (s,a) visited infinitely often
2. Learning rate $\alpha_t$ satisfies $\sum \alpha_t = \infty$, $\sum \alpha_t^2 < \infty$
   (e.g. $\alpha_t = 1/t$)

**For quadruped locomotion** (dog/robot dog like Boston Dynamics Spot):
- State: 12 joint angles + 12 velocities + body orientation + foot contacts = ~50 dims
- Action: 12 motor torques (continuous) → use actor-critic (PPO/SAC)
- Reward: forward velocity − energy − foot slip − fall penalty
'''))

# Policy gradient
display(Markdown(r'''
**Policy gradient theorem** (REINFORCE):
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ G_t \nabla_\theta \log \pi_\theta(a_t|s_t) \right]$$

**PPO clip objective** (used in quadruped RL):
$$L^{CLIP}(\theta) = \mathbb{E}\left[ \min\left( r_t(\theta)\hat{A}_t,\ \text{clip}(r_t, 1-\epsilon, 1+\epsilon)\hat{A}_t \right) \right]$$
where $r_t = \pi_\theta(a|s)/\pi_{\theta_{old}}(a|s)$, $\epsilon=0.2$ typical.
'''))

## §5 — PID Controller: Classical Feedback

In [ ]:
section('PID Control')

t, s = symbols('t s', real=True)
Kp, Ki, Kd = symbols('K_p K_i K_d', positive=True)
e = Function('e')(t)  # error = setpoint - measurement

# PID output
u = Kp*e + Ki*Integral(e, t) + Kd*Derivative(e, t)
step('PID control law u(t) =', u)

# Transfer function
C_pid = Kp + Ki/s + Kd*s
step('PID transfer function C(s) =', C_pid)
step('= Combined:', together(C_pid))

# Closed-loop with plant G(s) = 1/(s(s+1))
G = 1/(s*(s+1))
T_cl = C_pid*G / (1 + C_pid*G)
step('Plant G(s) = 1/[s(s+1)]:', G)
step('Closed-loop T(s) = CG/(1+CG):', simplify(T_cl))

# Characteristic polynomial
char_poly = denom(simplify(T_cl))
step('Characteristic polynomial (stability → all roots LHP):', char_poly)

## §6 — State Space and Optimal Control (LQR)

In [ ]:
section('State Space / LQR')

display(Markdown(r'''
**State space model**:
$$\dot{\mathbf{x}} = A\mathbf{x} + B\mathbf{u}$$
$$\mathbf{y} = C\mathbf{x} + D\mathbf{u}$$

**LQR** minimizes the infinite-horizon quadratic cost:
$$J = \int_0^\infty \left( \mathbf{x}^T Q \mathbf{x} + \mathbf{u}^T R \mathbf{u} \right) dt$$

Optimal gain: $\mathbf{u} = -K\mathbf{x}$, where $K = R^{-1}B^T P$
and $P$ solves the **algebraic Riccati equation**:
$$A^T P + P A - P B R^{-1} B^T P + Q = 0$$
'''))

# Simple 2D example: inverted pendulum
A = Matrix([[0, 1],[9.8, 0]])  # linearized inverted pendulum
B = Matrix([[0],[1]])
step('Inverted pendulum A =', A)
step('B =', B)
step('Open-loop eigenvalues (unstable!):', A.eigenvals())

# Controllability matrix
C_ctrl = B.row_join(A*B)
step('Controllability matrix [B, AB] =', C_ctrl)
step('rank = ', C_ctrl.rank(), ' (= n=2, so system is controllable)')

## §7 — Motor Driver Circuit: H-Bridge

In [ ]:
section('H-Bridge Motor Driver Circuit')

display(Markdown(r'''
```
         V_supply
           │
      ┌────┴────┐
     Q1(P)    Q3(P)    ← high-side MOSFETs
      │          │
     OUT_A    OUT_B  ─── Motor ───
      │          │
     Q2(N)    Q4(N)    ← low-side MOSFETs
      └────┬────┘
          GND

Forward:  Q1,Q4 ON  → Current A→B
Reverse:  Q2,Q3 ON  → Current B→A
Brake:    Q2,Q4 ON  → Motor shorted to GND
Coast:    All OFF
```

**PWM speed control**: switch Q1/Q4 at frequency $f_{PWM}$ (typically 20 kHz).
Average voltage = $V_{supply} \times$ duty cycle $D$.
'''))

V_s, D_cycle, R_m, L_m = symbols('V_s D R_m L_m', positive=True)
V_avg = V_s * D_cycle
step('Average motor voltage V_avg = V_s · D =', V_avg)

# Motor electrical model: V = L di/dt + R·i + back-EMF
omega_m, K_e = symbols('omega_m K_e', positive=True)
back_emf = K_e * omega_m
step('Back-EMF = K_e · ω =', back_emf)
step('Motor circuit: V_avg = L·di/dt + R·i + K_e·ω')

# Steady-state current
i_ss = (V_avg - back_emf) / R_m
step('Steady-state current i_ss = (V_avg − K_e·ω)/R =', i_ss)

# Torque
K_t = symbols('K_t', positive=True)
tau_motor = K_t * i_ss
step('Motor torque τ = K_t · i =', tau_motor)

## §8 — Full Loop: RL Policy → Controller → Circuit → Actuator

In [ ]:
section('Full System Loop')

display(Markdown(r'''
```
┌─────────────────────────────────────────────────────────────┐
│                     POLICY (RL / LQR)                       │
│  Input: state s = [q, q̇, orientation, contacts]            │
│  Output: action a = target joint torques τ_des              │
└───────────────────────┬─────────────────────────────────────┘
                        │ τ_des
                        ▼
┌─────────────────────────────────────────────────────────────┐
│                  JOINT-LEVEL PID                            │
│  Error: e = τ_des − τ_measured                              │
│  Output: PWM duty cycle D ∈ [0,1]                           │
└───────────────────────┬─────────────────────────────────────┘
                        │ D (PWM signal, 20 kHz)
                        ▼
┌─────────────────────────────────────────────────────────────┐
│              H-BRIDGE MOTOR DRIVER                          │
│  V_motor = V_supply × D                                     │
│  i_motor = (V_motor − K_e·ω) / R_m                         │
│  τ_output = K_t · i_motor                                   │
└───────────────────────┬─────────────────────────────────────┘
                        │ torque
                        ▼
┌─────────────────────────────────────────────────────────────┐
│              RIGID BODY DYNAMICS                            │
│  M(q)q̈ + C(q,q̇)q̇ + g(q) = τ                             │
│  → next state s'                                            │
└───────────────────────┬─────────────────────────────────────┘
                        │ s' (IMU + encoders)
                        └────────────────────► back to POLICY
```

**Timing**: RL policy runs at 50 Hz, PID at 1 kHz, PWM at 20 kHz, physics at ~1 MHz (simulation).
'''))

# Equations of motion summary
display(Markdown(r'''
**Manipulator equation of motion**:
$$M(q)\ddot{q} + C(q,\dot{q})\dot{q} + g(q) = \tau$$

- $M(q)$ — inertia matrix (symmetric positive definite)
- $C(q,\dot{q})$ — Coriolis/centrifugal matrix
- $g(q)$ — gravity vector
- $\tau$ — joint torques from actuators

For a 12-DOF quadruped: $M \in \mathbb{R}^{12\times 12}$, $\tau \in \mathbb{R}^{12}$.

**Connection to Lagrangian**:
$$M_{ij} = \frac{\partial^2 T}{\partial \dot{q}_i \partial \dot{q}_j}, \quad
g_i = \frac{\partial V}{\partial q_i}$$
'''))